# Text extraction (T21) — MarkItDown -> EasyOCR fallback, on real documents

Exercises `TextExtractor` (`classiflow.ingesta.extract`) against two real municipal
PDFs from `playground/samples/`: one with a real text layer (MarkItDown handles it
directly) and one that's a genuine scan with no text layer at all (MarkItDown yields
nothing, so it falls through to the EasyOCR + pymupdf rendering path).

Also demonstrates the individual `MarkItDownExtractor` / `OCRExtractor` classes on
their own — each is independently testable/usable, not just through the chain.
`OCRExtractor` takes its `easyocr.Reader` as a constructor argument (Container-managed
DI, not a hand-rolled singleton) — see `ingesta/extractors/ocr.py`.

> **Kernel**: select the project's `.venv` kernel in the top-right picker.
>
> **Heads up**: the OCR section auto-detects GPU (`torch.cuda.is_available()`) and
> falls back to CPU otherwise — CPU takes a few minutes for a multi-page scan, GPU
> is much faster.

## 1 — Imports

In [ ]:
from pathlib import Path

import easyocr

import classiflow
from classiflow.ingesta.config_extraction import get_extraction_config
from classiflow.ingesta.extract import TextExtractor
from classiflow.ingesta.extractors import MarkItDownExtractor, OCRExtractor
from classiflow.settings import Settings

_SAMPLES_DIR = Path(classiflow.__file__).parent / "playground" / "samples"

ocr_reader = easyocr.Reader([Settings.ocr_lang], gpu=True)
markitdown_extractor = MarkItDownExtractor()
ocr_extractor = OCRExtractor(reader=ocr_reader)
text_extractor = TextExtractor([markitdown_extractor, ocr_extractor])

_config = get_extraction_config()
print(f"min_text_for_ocr = {_config.min_text_for_ocr}  (below this, MarkItDown triggers OCR)")
print(f'min_usable_text  = {_config.min_usable_text}  (below this after OCR, returns "")')

## 2 — A real text-based PDF: MarkItDown handles it, OCR never runs

`convenio_2_2013.pdf` has a real embedded text layer — MarkItDown extracts it
directly and the result is already well above `MIN_TEXT_FOR_OCR`, so the OCR stage
in the chain never executes.

In [ ]:
text_pdf = (_SAMPLES_DIR / "convenio_2_2013.pdf").read_bytes()
print(f"file size: {len(text_pdf):,} bytes")

extraction = text_extractor(text_pdf, "convenio_2_2013.pdf")
text = extraction.text
print(f"\nextracted: {len(text):,} chars  (via {extraction.extractor_used})")
print(f"preview  : {text[:300]!r}")

## 3 — The individual extractor classes, directly

`TextExtractor` is a thin orchestrator over a chain of `ExtractorBase`
implementations — each one is usable (and testable) completely on its own. Reusing
the same `markitdown_extractor` instance built in section 1.

In [3]:
direct_text = markitdown_extractor.extract(text_pdf, "convenio_2_2013.pdf")
matches = direct_text == text
print(f"MarkItDownExtractor directly: {len(direct_text):,} chars")
print(f"matches TextExtractor() result: {matches}")

MarkItDownExtractor directly: 6,814 chars
matches TextExtractor() result: True


## 4 — A genuine scan: MarkItDown finds nothing, OCR does the real work

`boletin_65_2005_doc_39153.pdf` has no text layer at all — this is where the chain
actually falls through to `OCRExtractor`, which renders each page via pymupdf at
`Settings.ocr_render_dpi` (200) DPI and runs EasyOCR on the resulting image.

This cell takes a few minutes on CPU — that's genuinely how long real OCR inference
takes per page, not a notebook artifact.

In [ ]:
import time

scanned_pdf = (_SAMPLES_DIR / "boletin_65_2005_doc_39153.pdf").read_bytes()
print(f"file size: {len(scanned_pdf):,} bytes")

# Confirm MarkItDown alone really does come up empty on this one, before paying for OCR.
markitdown_only = markitdown_extractor.extract(scanned_pdf, "boletin_65_2005_doc_39153.pdf")
print(f"MarkItDown alone: {len(markitdown_only)} chars (below min_text_for_ocr -> OCR will run)")

start = time.monotonic()
extraction = text_extractor(scanned_pdf, "boletin_65_2005_doc_39153.pdf")
text = extraction.text
elapsed = time.monotonic() - start

print(f"\nOCR fallback took {elapsed:.1f}s")
print(f"extracted: {len(text):,} chars  (via {extraction.extractor_used})")
print(f"preview  : {text[:300]!r}")

## 5 — The OCR extractor directly, and a corrupt-input failure

Same `OCRExtractor` (and the same `reader`) used inside the chain above, called
directly — and what happens when it's handed something that isn't a valid PDF at
all: `pymupdf.FileDataError` is caught and re-raised as `classiflow`'s own
`OcrError`, not left as a bare pymupdf exception leaking out of this module.

In [5]:
from classiflow.ingesta.extractors.exceptions import OcrError

try:
    ocr_extractor.extract(b"this is not a pdf", "not-a-pdf.pdf")
except OcrError as exc:
    print(f"caught OcrError, as expected: {exc}")

caught OcrError, as expected: OCR failed for 'not-a-pdf.pdf': Failed to open stream


## 6 — Guardrail: TextExtractor never raises, always degrades to `""`

Feeding `text_extractor` itself the same garbage bytes — the chain catches the
`OcrError` internally and returns an empty string rather than propagating it, so a
single bad document can't crash the pipeline job that's ingesting it.

In [ ]:
result = text_extractor(b"this is not a pdf", "not-a-pdf.pdf")
print(f"result: {result!r}")
assert not result.text
print("confirmed: TextExtractor degrades gracefully instead of raising")